In [1]:
%%writefile app.py
import os
import tempfile
import streamlit as st

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama  # Local Ollama LLM

st.set_page_config(page_title="Mera AI App (Shiva)", layout="wide")
st.title("📄 Free AI Document Q&A App")

if "retriever" not in st.session_state:
    st.session_state.retriever = None

# Sidebar PDF Upload
with st.sidebar:
    st.header("Document Upload")
    uploaded_file = st.file_uploader("Apna PDF chunye", type=["pdf"])
    
    if uploaded_file and st.button("PDF Read Karo"):
        with st.spinner("AI PDF ko samajh raha hai..."):
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
                tmp_file.write(uploaded_file.read())
                tmp_path = tmp_file.name

            # PDF Load & Chunking
            loader = PyPDFLoader(tmp_path)
            documents = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
            chunks = text_splitter.split_documents(documents)

            # Local Embeddings & Vector DB
            embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
            vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)
            st.session_state.retriever = vector_db.as_retriever(search_kwargs={"k": 2})
            
            os.remove(tmp_path)
            st.success("✅ PDF Process Ho Gaya!")

# Question Answering Interface
if st.session_state.retriever:
    user_query = st.text_input("Apne document se jude sawaal poochhein:")
    if user_query:
        with st.spinner("Thinking..."):
            docs = st.session_state.retriever.invoke(user_query)
            context = "\n\n".join([doc.page_content for doc in docs])
            
            prompt = f"Context:\n{context}\n\nQuestion: {user_query}\nAnswer directly, accurately, and concisely based only on the context provided."
            
            try:
                # Local Ollama DeepSeek Model (No API Key Required!)
                llm = Ollama(model="deepseek-r1:8b")
                response = llm.invoke(prompt)
                
                st.markdown("### 💡 Jawab:")
                st.success(response.strip())
            except Exception as e:
                st.error(f"Error details: {e}\n\nMake sure Ollama is running in background!")
else:
    st.info("👈 Baayein (left) side se PDF upload karein.")

Overwriting app.py


In [ ]:
!streamlit run app.py